In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm # Para ver la barra de progreso

# 1. Configuración
dir_datos = "../../Datos/Datasets Finales"
archivo_onco = f"{dir_datos}/dataset_entrenamiento_onco_2019_2023.csv"
archivo_control = f"{dir_datos}/dataset_entrenamiento_control_2019_2023.csv"

# Parámetros del Bootstrapping
ITERACIONES = 50        # Cantidad de veces que repetiremos el experimento
TAMAÑO_MUESTRA = 20000  # Cuántos pacientes sacaremos por grupo en cada iteración
TARGET = 'MORTALIDAD'   # Variable objetivo a analizar

print("Cargando datasets de entrenamiento...")
df_onco = pd.read_csv(archivo_onco, low_memory=False)
df_control = pd.read_csv(archivo_control, low_memory=False)

# Separar características (X) y objetivo (y)
# Excluimos las otras variables objetivo para evitar sesgos directos entre ellas
cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER']
features = [col for col in df_onco.columns if col not in cols_excluir]

# Matriz para guardar las importancias de cada iteración
importancias_acumuladas = pd.DataFrame(index=features)

print(f"\nIniciando Bootstrapping ({ITERACIONES} iteraciones) para target: {TARGET}...")

for i in tqdm(range(ITERACIONES)):
    # 1. Muestreo Aleatorio (Bootstrapping con reemplazo)
    muestra_onco = df_onco.sample(n=TAMAÑO_MUESTRA, replace=True, random_state=i)
    muestra_control = df_control.sample(n=TAMAÑO_MUESTRA, replace=True, random_state=i)
    
    # Unir muestra balanceada
    df_balanceado = pd.concat([muestra_onco, muestra_control], ignore_index=True)
    
    X = df_balanceado[features]
    y = df_balanceado[TARGET]
    
    # 2. Entrenar modelo ligero para selección de características
    # Usamos max_depth=10 y pocos árboles para que sea muy rápido en cada iteración
    rf = RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1, random_state=i)
    rf.fit(X, y)
    
    # 3. Guardar importancia de esta iteración
    importancias_acumuladas[f'Iter_{i}'] = rf.feature_importances_

# 4. Calcular el promedio y desviación estándar de la importancia
importancias_acumuladas['Promedio'] = importancias_acumuladas.mean(axis=1)
importancias_acumuladas['Desv_Std'] = importancias_acumuladas.std(axis=1)

# Ordenar de mayor a menor importancia
top_features = importancias_acumuladas.sort_values(by='Promedio', ascending=False)

print("\nCálculo completado. Generando gráfico del Top 20 variables...")

# 5. Visualización del Top 20
top_20 = top_features.head(20)

plt.figure(figsize=(12, 8))
sns.barplot(x=top_20['Promedio'], y=top_20.index, xerr=top_20['Desv_Std'], color='steelblue')
plt.title(f"Top 20 Variables más importantes para {TARGET}\n(Promedio de {ITERACIONES} muestras balanceadas)")
plt.xlabel("Importancia Promedio (Random Forest)")
plt.ylabel("Variables (Incluyendo OHE)")
plt.tight_layout()

# Guardar y mostrar
ruta_grafico = f"../../Resultados/Resultados (etapa 1 y 2)/Análisis final/Importancia_{TARGET}.png"
plt.savefig(ruta_grafico, dpi=300)
plt.show()

# Exportar tabla completa
top_features[['Promedio', 'Desv_Std']].to_csv(f"../../Resultados/Resultados (etapa 1 y 2)/Análisis final/Ranking_Variables_{TARGET}.csv")
print(f"Resultados exportados a la carpeta de Análisis final.")